<a href="https://colab.research.google.com/github/christy5165/Denoising_Autoencoder.ipynb/blob/main/wk-10.A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# 1. Install/Update libraries
!pip install -q transformers[torch] datasets

import torch
from datasets import Dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    pipeline
)

# 2. Prepare Data
custom_text = [
    "The cosmic cat lived on the moon of Jupiter. It spent its days chasing stardust.",
    "The cosmic cat never slept because the stars were too bright and beautiful.",
    "Every time a comet passed, the cosmic cat would jump and catch a piece of ice.",
    "Galaxy travelers often saw the cosmic cat and offered it space-milk."
]
dataset = Dataset.from_dict({"text": custom_text})

# 3. Load Model and Tokenizer
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name)

# 4. Tokenize
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 5. Training Arguments (Fixed: Removed 'overwrite_output_dir')
training_args = TrainingArguments(
    output_dir="./gpt2-results",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    logging_steps=5,
    save_strategy="no", # Faster for a small test like this
    report_to="none"
)

# 6. Initialize Trainer and Train
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets,
)

print("Starting training...")
trainer.train()

# 7. Generate Results
print("\n--- GENERATED TEXT RESULT ---")
gen_pipeline = pipeline('text-generation', model=model, tokenizer=tokenizer)
prompt = "The cosmic cat"
result = gen_pipeline(prompt, max_length=50, do_sample=True, truncation=True)

print(result[0]['generated_text'])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Starting training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,4.003574
10,2.101338
15,1.525556
20,1.165587


Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- GENERATED TEXT RESULT ---
The cosmic cat lived on the moon. It was a huge cat. It had a belly and a tail. It hung on top of a tree and ate. It was big. It looked like a cat.

But it was just a cat. It slept on the tree. It didn't see things. It didn't notice the moon. It didn't know where the stars were. It didn't know what stars were. It didn't know what they were. It was a cat. It couldn't see anything. It couldn't see stars. It couldn't see stars. It couldn't see stars. It couldn't see stars. It couldn't see stars. It couldn't see. It couldn't see stars.

It couldn't see stars. It couldn't see stars. It couldn't see stars. It couldn't see stars.

It couldn't see stars. It couldn't see stars. It couldn't see stars.

It couldn't see stars. It couldn't see stars. It couldn't see stars.

It couldn't see stars. It couldn't see stars. It couldn't see stars.

It couldn't see stars. It couldn't see stars. It couldn't see stars.

It couldn't see
